<a href="https://colab.research.google.com/github/SanjayKesavan7/fst/blob/main/llm_lab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Foundations to Agents — Intensive 3-Hour Lab
**Hardware:** NVIDIA T4 GPU (16 GB VRAM) &nbsp;|&nbsp; **Level:** Beginner–Intermediate

| # | Section | Mode | Time |
|---|---|---|---|
| 1 | Foundations | 👀 Run & Observe | 45 min |
| 2 | Quantization | ✏️ Fill config fields | 30 min |
| 3 | Inference & Prompting | ✏️ Set params & run | 30 min |
| 4 | RAG with FAISS | ✏️ Wire pipeline steps | 45 min |
| 5 | Agents with CrewAI | ✏️ Define agents & tasks | 30 min |

> **Convention:** Every blank you must fill is marked `# TODO`.


In [1]:
# @title ⚙️ Setup — Run First (takes ~2 min)
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-gpu sentence-transformers
!pip install -q crewai crewai-tools
!pip install -q ctransformers huggingface_hub datasets
!pip install -q torch torchvision
print("\n✅ All packages installed.")
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch runtime to T4"
print(f"GPU: {gpu}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 👀 Section 1: Foundations — Run & Observe
**45 min | Tasks 1–5** — No blanks. Read each cell, run it, understand the output.


In [6]:
# @title Task 1 — BPE: How tokenizers build their vocabulary
# BPE starts with characters and iteratively merges the most frequent adjacent pair.
from collections import defaultdict

def get_vocab(corpus):
    vocab = defaultdict(int)
    for word in corpus:
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[(syms[i], syms[i+1])] += freq
    return dict(pairs)

def merge_vocab(pair, vocab):
    import re
    bigram  = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    return {pattern.sub(''.join(pair), w): f for w, f in vocab.items()}

corpus = ['low','lower','newest','widest','low','low']
vocab  = get_vocab(corpus)
print("Initial vocab:", vocab)
for i in range(4):
    best = max(get_stats(vocab), key=get_stats(vocab).get)
    vocab = merge_vocab(best, vocab)
    print(f'Merge {i+1}: {"+".join(best)} -> {"".join(best)}')
print("\nFinal vocab:", vocab)

Initial vocab: {'l o w </w>': 3, 'l o w e r </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1}
Merge 1: l+o -> lo
Merge 2: lo+w -> low
Merge 3: low+</w> -> low</w>
Merge 4: e+s -> es

Final vocab: {'low</w>': 3, 'low e r </w>': 1, 'n e w es t </w>': 1, 'w i d es t </w>': 1}


In [53]:
# @title Task 1 — BPE: How tokenizers build their vocabulary
# BPE starts with characters and iteratively merges the most frequent adjacent pair.
from collections import defaultdict

def get_vocab(corpus):
    vocab = defaultdict(int)
    for word in corpus:
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[(syms[i], syms[i+1])] += freq
    return dict(pairs)

def merge_vocab(pair, vocab):
    import re
    bigram  = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    return {pattern.sub(''.join(pair), w): f for w, f in vocab.items()}

corpus = ['low','lower','newest','widest','low','low']
vocab  = get_vocab(corpus)
print("Initial vocab:", vocab)
for i in range(4):
    best = max(get_stats(vocab), key=get_stats(vocab).get)
    vocab = merge_vocab(best, vocab)
    print(f'Merge {i+1}: {"+".join(best)} -> {"".join(best)}')
print("\nFinal vocab:", vocab)

Initial vocab: {'l o w </w>': 3, 'l o w e r </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1}
Merge 1: l+o -> lo
Merge 2: lo+w -> low
Merge 3: low+</w> -> low</w>
Merge 4: e+s -> es

Final vocab: {'low</w>': 3, 'low e r </w>': 1, 'n e w es t </w>': 1, 'w i d es t </w>': 1}


In [54]:
# @title Task 2 — Softmax: turning logits into probabilities
import numpy as np

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)   # subtract max for numerical stability
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

logits = np.array([2.0, 1.0, 0.1, -1.0])
probs  = softmax(logits)
print(f"Logits: {logits}")
print(f"Probs:  {probs.round(4)}")
print(f"Sum:    {probs.sum():.6f}  <- must be 1.0")

Logits: [ 2.   1.   0.1 -1. ]
Probs:  [0.6381 0.2347 0.0954 0.0318]
Sum:    1.000000  <- must be 1.0


In [55]:
# @title Task 3 — Scaled Dot-Product Attention
import torch, math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k    = Q.size(-1)
    scores = (Q @ K.transpose(-2,-1)) / math.sqrt(d_k)   # scale
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn

B, S, dk = 2, 5, 64
Q, K, V  = torch.randn(B,S,dk), torch.randn(B,S,dk), torch.randn(B,S,dk)
out, aw  = scaled_dot_product_attention(Q, K, V)
print(f"Output shape      : {out.shape}")
print(f"Attn weight shape : {aw.shape}")
print(f"Row-0 sum         : {aw[0,0].sum().item():.6f}  <- must be 1.0")

Output shape      : torch.Size([2, 5, 64])
Attn weight shape : torch.Size([2, 5, 5])
Row-0 sum         : 1.000000  <- must be 1.0


In [56]:
# @title Task 4 — MoE: Top-1 Expert Routing
import torch, torch.nn as nn

class MoETop1Router(nn.Module):
    def __init__(self, d_model, num_experts):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
    def forward(self, x):
        probs   = torch.softmax(self.gate(x), dim=-1)
        indices = probs.argmax(dim=-1)          # pick best expert per token
        return indices, probs

router = MoETop1Router(128, 8)
x      = torch.randn(2, 10, 128)  # batch=2, seq=10
ei, rp = router(x)
print(f"Expert chosen per token (batch 0): {ei[0].tolist()}")
print(f"Router probs shape               : {rp.shape}")
print(f"Probs sum to 1? {rp.sum(-1).allclose(torch.ones(2,10), atol=1e-4)}")

Expert chosen per token (batch 0): [3, 1, 4, 4, 1, 6, 3, 2, 6, 7]
Router probs shape               : torch.Size([2, 10, 8])
Probs sum to 1? True


In [57]:
# @title Task 5 — Transformer Forward Pass via HuggingFace (Demo)
# See the full pipeline in one call — tokenise → embed → attend → logits
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok   = AutoTokenizer.from_pretrained('facebook/opt-125m')
model = AutoModelForCausalLM.from_pretrained('facebook/opt-125m')
model.eval()

inputs = tok('The capital of France is', return_tensors='pt')
with torch.no_grad():
    out = model(**inputs)
next_token_id   = out.logits[0, -1].argmax().item()
next_token_text = tok.decode([next_token_id])
print(f"Logits shape : {out.logits.shape}  (batch, seq_len, vocab_size)")
print(f"Next token   : '{next_token_text}'")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Logits shape : torch.Size([1, 6, 50272])  (batch, seq_len, vocab_size)
Next token   : ' the'


In [59]:
# @title 🔍 Section 1 Grader { display-mode: "form" }
import numpy as np, torch

_s1_passed = _s1_total = 0

def run_test(label, fn):
    global _s1_passed, _s1_total
    _s1_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s1_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 1 — get_stats')
def _t1a():
    s = get_stats({'a b </w>':3, 'a b c </w>':2})
    assert s.get(('a','b'),0)==5, f"expected 5, got {s.get(('a','b'),0)}"
def _t1b():
    s = get_stats({'x y </w>':4})
    assert s.get(('x','y'),0)==4 and s.get(('y','</w>'),0)==4
run_test('pair (a,b) count=5 across two vocab entries', _t1a)
run_test('single entry: all pairs inherit word frequency', _t1b)

print('Task 2 — softmax')
def _t2a():
    o = softmax(np.array([0.,0.,0.]))
    assert np.allclose(o,[1/3,1/3,1/3],atol=1e-5), f'got {o}'
def _t2b():
    o = softmax(np.array([1000.,1001.,1002.]))
    assert not np.any(np.isnan(o)), 'NaN — subtract max missing'
    assert o[2]>o[1]>o[0], 'ordering wrong'
run_test('uniform input -> [1/3, 1/3, 1/3]', _t2a)
run_test('large values: no NaN, ordering preserved', _t2b)

print('Task 3 — attention')
def _t3a():
    Q,K,V = torch.randn(2,4,16),torch.randn(2,4,16),torch.randn(2,4,16)
    o,aw = scaled_dot_product_attention(Q,K,V)
    assert tuple(o.shape)==(2,4,16), f'shape {o.shape}'
    assert torch.allclose(aw.sum(-1),torch.ones(2,4),atol=1e-4)
def _t3b():
    mask = torch.tensor([[[False,True],[False,False]]])
    Q,K,V = torch.randn(1,2,8),torch.randn(1,2,8),torch.randn(1,2,8)
    _,aw = scaled_dot_product_attention(Q,K,V,mask=mask)
    assert aw[0,0,1].item()<1e-6, f'masked weight={aw[0,0,1].item():.6f}'
run_test('output shape (2,4,16) and attn rows sum to 1', _t3a)
run_test('masked position receives ~0 attention weight', _t3b)

print('Task 4 — MoE router')
def _t4a():
    r = MoETop1Router(32,6); ei,rp = r(torch.randn(2,5,32))
    assert tuple(ei.shape)==(2,5) and tuple(rp.shape)==(2,5,6)
    assert torch.allclose(rp.sum(-1),torch.ones(2,5),atol=1e-4)
def _t4b():
    r = MoETop1Router(16,4); ei,rp = r(torch.randn(1,3,16))
    assert (ei>=0).all() and (ei<4).all(), f'index out of range: {ei}'
    assert torch.equal(ei, rp.argmax(-1)), 'indices must match argmax of probs'
run_test('shapes correct and router_probs sum to 1', _t4a)
run_test('indices in [0,num_experts) and match argmax', _t4b)

print(f"\nSection 1: {_s1_passed}/{_s1_total} tests passed")

Task 1 — get_stats
  ✅ PASS  pair (a,b) count=5 across two vocab entries
  ✅ PASS  single entry: all pairs inherit word frequency
Task 2 — softmax
  ✅ PASS  uniform input -> [1/3, 1/3, 1/3]
  ✅ PASS  large values: no NaN, ordering preserved
Task 3 — attention
  ✅ PASS  output shape (2,4,16) and attn rows sum to 1
  ✅ PASS  masked position receives ~0 attention weight
Task 4 — MoE router
  ✅ PASS  shapes correct and router_probs sum to 1
  ✅ PASS  indices in [0,num_experts) and match argmax

Section 1: 8/8 tests passed


In [24]:
# @title Task 2 — Softmax: turning logits into probabilities
import numpy as np

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)   # subtract max for numerical stability
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

logits = np.array([2.0, 1.0, 0.1, -1.0])
probs  = softmax(logits)
print(f"Logits: {logits}")
print(f"Probs:  {probs.round(4)}")
print(f"Sum:    {probs.sum():.6f}  <- must be 1.0")

Logits: [ 2.   1.   0.1 -1. ]
Probs:  [0.6381 0.2347 0.0954 0.0318]
Sum:    1.000000  <- must be 1.0


In [ ]:
# @title Task 3 — Scaled Dot-Product Attention
import torch, math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k    = Q.size(-1)
    scores = (Q @ K.transpose(-2,-1)) / math.sqrt(d_k)   # scale
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn

B, S, dk = 2, 5, 64
Q, K, V  = torch.randn(B,S,dk), torch.randn(B,S,dk), torch.randn(B,S,dk)
out, aw  = scaled_dot_product_attention(Q, K, V)
print(f"Output shape      : {out.shape}")
print(f"Attn weight shape : {aw.shape}")
print(f"Row-0 sum         : {aw[0,0].sum().item():.6f}  <- must be 1.0")

In [ ]:
# @title Task 4 — MoE: Top-1 Expert Routing
import torch, torch.nn as nn

class MoETop1Router(nn.Module):
    def __init__(self, d_model, num_experts):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
    def forward(self, x):
        probs   = torch.softmax(self.gate(x), dim=-1)
        indices = probs.argmax(dim=-1)          # pick best expert per token
        return indices, probs

router = MoETop1Router(128, 8)
x      = torch.randn(2, 10, 128)  # batch=2, seq=10
ei, rp = router(x)
print(f"Expert chosen per token (batch 0): {ei[0].tolist()}")
print(f"Router probs shape               : {rp.shape}")
print(f"Probs sum to 1? {rp.sum(-1).allclose(torch.ones(2,10), atol=1e-4)}")

In [ ]:
# @title Task 5 — Transformer Forward Pass via HuggingFace (Demo)
# See the full pipeline in one call — tokenise → embed → attend → logits
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok   = AutoTokenizer.from_pretrained('facebook/opt-125m')
model = AutoModelForCausalLM.from_pretrained('facebook/opt-125m')
model.eval()

inputs = tok('The capital of France is', return_tensors='pt')
with torch.no_grad():
    out = model(**inputs)
next_token_id   = out.logits[0, -1].argmax().item()
next_token_text = tok.decode([next_token_id])
print(f"Logits shape : {out.logits.shape}  (batch, seq_len, vocab_size)")
print(f"Next token   : '{next_token_text}'")

In [23]:
# @title 🔍 Section 1 Grader { display-mode: "form" }
import numpy as np, torch

_s1_passed = _s1_total = 0

def run_test(label, fn):
    global _s1_passed, _s1_total
    _s1_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s1_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 1 — get_stats')
def _t1a():
    s = get_stats({'a b </w>':3, 'a b c </w>':2})
    assert s.get(('a','b'),0)==5, f"expected 5, got {s.get(('a','b'),0)}"
def _t1b():
    s = get_stats({'x y </w>':4})
    assert s.get(('x','y'),0)==4 and s.get(('y','</w>'),0)==4
run_test('pair (a,b) count=5 across two vocab entries', _t1a)
run_test('single entry: all pairs inherit word frequency', _t1b)

print('Task 2 — softmax')
def _t2a():
    o = softmax(np.array([0.,0.,0.]))
    assert np.allclose(o,[1/3,1/3,1/3],atol=1e-5), f'got {o}'
def _t2b():
    o = softmax(np.array([1000.,1001.,1002.]))
    assert not np.any(np.isnan(o)), 'NaN — subtract max missing'
    assert o[2]>o[1]>o[0], 'ordering wrong'
run_test('uniform input -> [1/3, 1/3, 1/3]', _t2a)
run_test('large values: no NaN, ordering preserved', _t2b)

print('Task 3 — attention')
def _t3a():
    Q,K,V = torch.randn(2,4,16),torch.randn(2,4,16),torch.randn(2,4,16)
    o,aw = scaled_dot_product_attention(Q,K,V)
    assert tuple(o.shape)==(2,4,16), f'shape {o.shape}'
    assert torch.allclose(aw.sum(-1),torch.ones(2,4),atol=1e-4)
def _t3b():
    mask = torch.tensor([[[False,True],[False,False]]])
    Q,K,V = torch.randn(1,2,8),torch.randn(1,2,8),torch.randn(1,2,8)
    _,aw = scaled_dot_product_attention(Q,K,V,mask=mask)
    assert aw[0,0,1].item()<1e-6, f'masked weight={aw[0,0,1].item():.6f}'
run_test('output shape (2,4,16) and attn rows sum to 1', _t3a)
run_test('masked position receives ~0 attention weight', _t3b)

print('Task 4 — MoE router')
def _t4a():
    r = MoETop1Router(32,6); ei,rp = r(torch.randn(2,5,32))
    assert tuple(ei.shape)==(2,5) and tuple(rp.shape)==(2,5,6)
    assert torch.allclose(rp.sum(-1),torch.ones(2,5),atol=1e-4)
def _t4b():
    r = MoETop1Router(16,4); ei,rp = r(torch.randn(1,3,16))
    assert (ei>=0).all() and (ei<4).all(), f'index out of range: {ei}'
    assert torch.equal(ei, rp.argmax(-1)), 'indices must match argmax of probs'
run_test('shapes correct and router_probs sum to 1', _t4a)
run_test('indices in [0,num_experts) and match argmax', _t4b)

print(f"\nSection 1: {_s1_passed}/{_s1_total} tests passed")

Task 1 — get_stats
  ❌ FAIL  pair (a,b) count=5 across two vocab entries
         -> name 'get_stats' is not defined
  ❌ FAIL  single entry: all pairs inherit word frequency
         -> name 'get_stats' is not defined
Task 2 — softmax
  ❌ FAIL  uniform input -> [1/3, 1/3, 1/3]
         -> name 'softmax' is not defined
  ❌ FAIL  large values: no NaN, ordering preserved
         -> name 'softmax' is not defined
Task 3 — attention
  ❌ FAIL  output shape (2,4,16) and attn rows sum to 1
         -> name 'scaled_dot_product_attention' is not defined
  ❌ FAIL  masked position receives ~0 attention weight
         -> name 'scaled_dot_product_attention' is not defined
Task 4 — MoE router
  ❌ FAIL  shapes correct and router_probs sum to 1
         -> name 'MoETop1Router' is not defined
  ❌ FAIL  indices in [0,num_experts) and match argmax
         -> name 'MoETop1Router' is not defined

Section 1: 0/8 tests passed


---
## ✏️ Section 2: Quantization
**30 min | Tasks 6–10** — Configure BitsAndBytes. Fill every `# TODO`.


In [28]:
def compute_model_memory_gb(num_params_millions, dtype_bytes):
    """
    Args:
        num_params_millions : model size in millions (e.g. 7000 for a 7B model)
        dtype_bytes         : 4=FP32, 2=FP16/BF16, 1=INT8, 0.5=INT4
    Returns: memory in GB (float)
    """
    # Use the formula: params * 10^6 * bytes / 10^9
    memory_GB = (num_params_millions * 1e6 * dtype_bytes) / 1e9
    return memory_GB

# --- DO NOT FORGET TO RUN THIS CELL ---

In [8]:
# @title Task 7 — Configure 4-bit NF4 Quantization
from transformers import BitsAndBytesConfig
import torch

# TODO: Create a BitsAndBytesConfig named bnb_config with:

bnb_config = BitsAndBytesConfig(load_in_4bit=True,
  bnb_4bit_quant_type='nf4',
  bnb_4bit_compute_dtype=torch.float16,
  bnb_4bit_use_double_quant=False)

print('BitsAndBytes config created:', bnb_config)

BitsAndBytes config created: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [9]:
# @title Task 8 — Load OPT-125M in 4-bit and measure VRAM
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'facebook/opt-125m'

torch.cuda.empty_cache()
vram_before = torch.cuda.memory_reserved(0) / 1e9

# TODO: Load model using AutoModelForCausalLM.from_pretrained with:
model_4bit = AutoModelForCausalLM.from_pretrained(MODEL_ID,
quantization_config = bnb_config,
device_map = 'auto'
)  # replace with from_pretrained call

vram_after = torch.cuda.memory_reserved(0) / 1e9
print(f"VRAM before: {vram_before:.3f} GB")
print(f"VRAM after : {vram_after:.3f} GB")
print(f"Model used : {vram_after - vram_before:.3f} GB")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


VRAM before: 0.000 GB
VRAM after : 0.346 GB
Model used : 0.346 GB


In [10]:
# @title Task 9 — Enable Double Quantization
# Double quantization also quantizes the quantization constants,
# saving an additional ~0.37 bits/param (from the QLoRA paper).

# TODO: Create bnb_config_dq — identical to bnb_config but with
#   bnb_4bit_use_double_quant=True
bnb_config_dq = BitsAndBytesConfig(load_in_4bit=True,
  bnb_4bit_quant_type='nf4',
  bnb_4bit_compute_dtype=torch.float16,
  bnb_4bit_use_double_quant=True)    # replace

print(f"Standard NF4 double_quant: {bnb_config.bnb_4bit_use_double_quant}")
print(f"Double quant config      : {bnb_config_dq.bnb_4bit_use_double_quant}")

Standard NF4 double_quant: False
Double quant config      : True


In [22]:
# @title Task 10 — Inspect Quantized Layer Dtypes
# After loading a 4-bit model, check that weights are actually stored in 4-bit.
# Iterate over the model's parameters
for i, (name, param) in enumerate(model_4bit.named_parameters()):
    # Stop after the first 8
    if i >= 8:
        break

    print(f"Parameter: {name} | Dtype: {param.dtype}")
# TODO: iterate over model_4bit.named_parameters()
#   print the name and dtype of the first 8 parameters
#   hint: for name, param in model_4bit.named_parameters():
pass

# Expected: you should see dtype=torch.uint8 for quantized layers

Parameter: model.decoder.embed_tokens.weight | Dtype: torch.float16
Parameter: model.decoder.embed_positions.weight | Dtype: torch.float16
Parameter: model.decoder.final_layer_norm.weight | Dtype: torch.float16
Parameter: model.decoder.final_layer_norm.bias | Dtype: torch.float16
Parameter: model.decoder.layers.0.self_attn.k_proj.weight | Dtype: torch.uint8
Parameter: model.decoder.layers.0.self_attn.k_proj.bias | Dtype: torch.float16
Parameter: model.decoder.layers.0.self_attn.v_proj.weight | Dtype: torch.uint8
Parameter: model.decoder.layers.0.self_attn.v_proj.bias | Dtype: torch.float16


In [29]:
# @title 🔍 Section 2 Grader { display-mode: "form" }
import torch
from transformers import BitsAndBytesConfig

_s2_passed = _s2_total = 0

def run_test(label, fn):
    global _s2_passed, _s2_total
    _s2_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s2_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 6 — memory math')
def _t6a():
    assert abs(compute_model_memory_gb(125,2)-0.25)<0.01, f'OPT-125M FP16 should be 0.25GB'
def _t6b():
    assert abs(compute_model_memory_gb(7000,0.5)-3.5)<0.1, f'7B INT4 should be ~3.5GB'
run_test('OPT-125M FP16 = 0.25 GB', _t6a)
run_test('LLaMA-7B INT4 = ~3.5 GB', _t6b)

print('Task 7 — bnb_config')
def _t7a():
    assert bnb_config is not None and isinstance(bnb_config, BitsAndBytesConfig)
    assert bnb_config.load_in_4bit == True
def _t7b():
    assert bnb_config.bnb_4bit_quant_type == 'nf4', f'got {bnb_config.bnb_4bit_quant_type}'
    assert bnb_config.bnb_4bit_compute_dtype == torch.float16
run_test('load_in_4bit=True and correct type', _t7a)
run_test('quant_type=nf4, compute_dtype=float16', _t7b)

print('Task 8 — model_4bit loaded')
def _t8a():
    assert model_4bit is not None, 'model_4bit is None'
def _t8b():
    dtypes = {p.dtype for _,p in model_4bit.named_parameters()}
    assert torch.uint8 in dtypes, f'No uint8 params found — model may not be 4-bit. Dtypes: {dtypes}'
run_test('model_4bit is not None', _t8a)
run_test('at least one uint8 (4-bit) layer present', _t8b)

print('Task 9 — double quant config')
def _t9a():
    assert bnb_config_dq is not None and bnb_config_dq.bnb_4bit_use_double_quant == True
def _t9b():
    assert bnb_config_dq.bnb_4bit_quant_type == 'nf4'
    assert bnb_config.bnb_4bit_use_double_quant == False, 'original config must stay False'
run_test('double_quant=True in new config', _t9a)
run_test('quant_type still nf4, original unchanged', _t9b)

print(f"\nSection 2: {_s2_passed}/{_s2_total} tests passed")

Task 6 — memory math
  ✅ PASS  OPT-125M FP16 = 0.25 GB
  ✅ PASS  LLaMA-7B INT4 = ~3.5 GB
Task 7 — bnb_config
  ✅ PASS  load_in_4bit=True and correct type
  ✅ PASS  quant_type=nf4, compute_dtype=float16
Task 8 — model_4bit loaded
  ✅ PASS  model_4bit is not None
  ✅ PASS  at least one uint8 (4-bit) layer present
Task 9 — double quant config
  ✅ PASS  double_quant=True in new config
  ✅ PASS  quant_type still nf4, original unchanged

Section 2: 8/8 tests passed


---
## ✏️ Section 3: Inference & Prompting
**30 min | Tasks 11–15** — Load a model, tune decoding parameters, and engineer prompts.


In [1]:
from huggingface_hub import hf_hub_download
from ctransformers import AutoModelForCausalLM as CT_Model

REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'
FILE = 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf'

# This downloads the specific GGUF file and returns the local path string
gguf_path = hf_hub_download(repo_id=REPO, filename=FILE)

# TODO: Load the GGUF model with CT_Model.from_pretrained.
# Note: We pass the local path to the .gguf file as the first argument.
gguf_model = CT_Model.from_pretrained(
    gguf_path,
    model_type='llama',
    gpu_layers=50
)

# Verify
out = gguf_model('Hello!', max_new_tokens=10)
print(f"Model loaded. Quick test: {out[:80]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded. Quick test:  This is my first time here and I want to


In [2]:
# @title Task 12 — Explore Temperature: how it changes response style
# Temperature controls randomness.
#   Low  (0.1–0.4) -> focused, repetitive, deterministic
#   Mid  (0.7–1.0) -> balanced
#   High (1.2–2.0) -> creative, unpredictable, sometimes incoherent

PROMPT = '<|system|>\nYou are a helpful assistant.\n<|user|>\nDescribe the ocean in one sentence.\n<|assistant|>\n'

# TODO: Choose three temperature values to compare. Fill in the list below.
#   Pick one low (<0.4), one mid (~0.8), one high (>1.2)
# TODO: Choose three temperature values to compare. Fill in the list below.
#   Pick one low (<0.4), one mid (~0.8), one high (>1.2)
temperatures_to_try = [0.2, 0.8, 1.5]

for temp in temperatures_to_try:
    # We call the model with the specific temperature for this loop iteration
    response = gguf_model(PROMPT, max_new_tokens=60, temperature=temp, repetition_penalty=1.1)
    print(f'--- temperature={temp} ---')
    print(response.strip()[:200])
    print()

--- temperature=0.2 ---
The Oceania region is a vast and diverse ecosystem, characterized by its unique flora and fauna.

--- temperature=0.8 ---
Oceans are awe-inspiring, vast and ever-changing entities filled with treasures of endless mysteries.

--- temperature=1.5 ---
The ocean is a vast and captivating environment where vast currents crash to foothills rising majestically at a distance.



In [3]:
# @title Task 13 — Explore Top-P and Top-K Sampling
# top_p (nucleus sampling): keep smallest set of tokens whose cumulative prob >= p
#   top_p=1.0  -> use all tokens (no filtering)
#   top_p=0.5  -> only top 50% probability mass
# top_k: keep only the k highest-probability tokens
#   top_k=0    -> disabled
#   top_k=40   -> sample from top 40 tokens only

PROMPT2 = '<|system|>\nYou are a creative storyteller.\n<|user|>\nBegin a story: Describe you\n<|assistant|>\n'

# TODO: Fill in the experiment table below with values you want to explore.
#   Each row is a dict with keys: label, top_p, top_k
# TODO: Fill in the experiment table below with values you want to explore.
# Each row MUST be a dict with keys: 'label', 'top_p', and 'top_k'
experiments = [
    {'label': 'wide nucleus (creative)',    'top_p': 0.95, 'top_k': 0  },
    {'label': 'narrow nucleus (focused)',   'top_p': 0.3,  'top_k': 0  },
    {'label': 'top-k only (diverse)',       'top_p': 1.0,  'top_k': 50 },
    {'label': 'strict top-k (limited)',      'top_p': 1.0,  'top_k': 5  },
    {'label': 'hybrid (balanced)',          'top_p': 0.9,  'top_k': 40 }
]

for exp in experiments:
    # We unpack the dictionary values into the model call
    response = gguf_model(
        PROMPT2,
        max_new_tokens=80,
        temperature=0.8,
        top_p=exp['top_p'],
        top_k=exp['top_k'],
        repetition_penalty=1.1
    )
    print(f'--- {exp["label"]} (top_p={exp["top_p"]}, top_k={exp["top_k"]}) ---')
    print(response.strip())
    print()

--- wide nucleus (creative) (top_p=0.95, top_k=0) ---
I am a solitary figure, with no family or friends to speak of. I have always been an introverted person, preferring solitude over social interaction. However, my life took a turn when I stumbled upon a mysterious book in the library. It was a collection of stories, each one more intriguing than the last.

As I flipped through the

--- narrow nucleus (focused) (top_p=0.3, top_k=0) ---
I am a solitary figure, with no family or friends to speak of. I have always been an introverted person, preferring solitude over socializing. However, my life took a turn when I stumbled upon a mysterious book in the library. It was a collection of stories, each one more intriguing than the last.

As I flipped through the

--- top-k only (diverse) (top_p=1.0, top_k=50) ---
I am a young woman, with curly hair that I always seem to have a knot in and vivid blue eyes that sparkle like the sun at dawn. I'm tall for my age, standing on my tiptoes like a cat

In [4]:
# @title Task 14 — Few-Shot Prompting
# Few-shot prompting provides examples to help the model learn a pattern.

# TODO: Create a few-shot prompt with at least 3 examples.
# Make sure the last one is an "Input:" without an "Output:" yet.
few_shot_prompt = """
Input: The movie was amazing and I loved the acting!
Output: Positive

Input: The food was cold and the service was terrible.
Output: Negative

Input: It was an okay experience, nothing special.
Output: Neutral

Input: I can't believe how fast the shipping was; it arrived early!
Output:"""

# Run the prompt through your GGUF model
# Note: stop=['\n'] prevents the model from rambling into new examples
response = gguf_model(few_shot_prompt, max_new_tokens=10, stop=['\n'])
print(f"Few-Shot Result: {response.strip()}")

Few-Shot Result: Positive


In [8]:
# @title Task 15 — Chain-of-Thought Prompting
# CoT: appending 'Let\'s think step by step.' dramatically improves reasoning.

QUESTION = 'A train travels 120 km in 1.5 hours, stops for 20 minutes, then covers 90 km in 1 hour. What is the average speed for the entire journey?'

# TODO: Build cot_prompt in ChatML format:
#   <|system|>
#   You are a precise reasoning assistant.
#   <|user|>
#   {QUESTION}
#   Let's think step by step.
#   <|assistant|>
cot_prompt = f"""<|system|>
You are a precise reasoning assistant.
<|user|>
{QUESTION}
Let's think step by step.
<|assistant|>
"""  # TODO: fill in the full prompt string

response = gguf_model(cot_prompt, max_new_tokens=200, temperature=0.3)
print(response)

The train travels a distance of 120 km during the first 1.5 hours, which means it has covered 60 km at an average speed of 8 km/hr.
During the second hour, the train stops for 20 minutes, so its speed decreases to 4 km/hr.
Finally, after covering 90 km in 1 hour, the train's speed is 3 km/hr.
Therefore, the average speed of the entire journey is:
(60 km / (8 km/hr + 4 km/hr)) * 2 = 3 km/hr
So, the average speed for the entire journey is 3 km/hr.


In [9]:
# @title 🔍 Section 3 Grader { display-mode: "form" }
_s3_passed = _s3_total = 0

def run_test(label, fn):
    global _s3_passed, _s3_total
    _s3_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s3_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 11 — GGUF model')
run_test('gguf_model is loaded', lambda: (_ for _ in ()).throw(AssertionError('gguf_model is None')) if gguf_model is None else None)
def _t11b():
    out = gguf_model('Test', max_new_tokens=3)
    assert isinstance(out, str) and len(out)>0
run_test('model generates non-empty string output', _t11b)

print('Task 12 — temperature exploration')
def _t12a():
    assert len(temperatures_to_try)>=2, f'need >=2 temps, got {len(temperatures_to_try)}'
def _t12b():
    assert all(isinstance(t,(int,float)) and t>0 for t in temperatures_to_try), 'all temps must be positive numbers'
    assert max(temperatures_to_try)!=min(temperatures_to_try), 'try distinct temperature values'
run_test('at least 2 temperature values defined', _t12a)
run_test('all values positive and distinct', _t12b)

print('Task 13 — top-p / top-k exploration')
def _t13a():
    assert len(experiments)>=2, f'define >=2 experiments, got {len(experiments)}'
def _t13b():
    for e in experiments:
        assert 'top_p' in e and 'top_k' in e, f'experiment missing top_p or top_k: {e}'
run_test('at least 2 sampling experiments defined', _t13a)
run_test('each experiment has top_p and top_k keys', _t13b)

print('Task 14 — few-shot prompt')
def _t14a():
    assert few_shot_prompt.count('Input:')>=3, f'need >=3 Input: lines, got {few_shot_prompt.count("Input:")}'
def _t14b():
    assert few_shot_prompt.count('Output:')>=3
    assert few_shot_prompt.strip().endswith('Output:'), 'prompt must end with Output: for the new query'
run_test('3+ examples: 3 Input: lines present', _t14a)
run_test('3+ Output: lines, last one is open-ended', _t14b)

print('Task 15 — CoT prompt')
def _t15a():
    assert '<|system|>' in cot_prompt and '<|user|>' in cot_prompt and '<|assistant|>' in cot_prompt
def _t15b():
    assert 'step by step' in cot_prompt.lower(), 'CoT trigger phrase missing'
    assert QUESTION[:20] in cot_prompt, 'question not in prompt'
run_test('ChatML tags present', _t15a)
run_test('CoT trigger phrase and question in prompt', _t15b)

print(f"\nSection 3: {_s3_passed}/{_s3_total} tests passed")

Task 11 — GGUF model
  ✅ PASS  gguf_model is loaded
  ✅ PASS  model generates non-empty string output
Task 12 — temperature exploration
  ✅ PASS  at least 2 temperature values defined
  ✅ PASS  all values positive and distinct
Task 13 — top-p / top-k exploration
  ✅ PASS  at least 2 sampling experiments defined
  ✅ PASS  each experiment has top_p and top_k keys
Task 14 — few-shot prompt
  ✅ PASS  3+ examples: 3 Input: lines present
  ✅ PASS  3+ Output: lines, last one is open-ended
Task 15 — CoT prompt
  ✅ PASS  ChatML tags present
  ✅ PASS  CoT trigger phrase and question in prompt

Section 3: 10/10 tests passed


---
## ✏️ Section 4: RAG with FAISS
**45 min | Tasks 16–20** — Build a full Retrieval-Augmented Generation pipeline.


In [15]:
# @title Task 16 — Load Web Documents with LangChain
from langchain_community.document_loaders import WebBaseLoader

URLS = [
    'https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)',
    'https://en.wikipedia.org/wiki/Large_language_model',
]

# TODO: Create a WebBaseLoader with the URLS list, then call .load()
docs = WebBaseLoader(URLS).load()

print(f"Documents loaded: {len(docs)}")
print(f"First doc length: {len(docs[0].page_content)} chars")

Documents loaded: 2
First doc length: 105356 chars


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Your existing code should work now:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks = text_splitter.split_documents(docs)

print(f"Chunks produced : {len(chunks)}")
print(f"Sample chunk    : {chunks[0].page_content[:150]}...")

Chunks produced : 665
Sample chunk    : Transformer (deep learning) - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Naviga...


In [17]:
# @title Task 18 — Create Sentence Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# TODO: Instantiate HuggingFaceEmbeddings with:
#   model_name=EMBED_MODEL
#   model_kwargs={'device': 'cuda'}
#   encode_kwargs={'normalize_embeddings': True}
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)  # replace

vec = embeddings.embed_query('What is attention?')
print(f"Embedding dim: {len(vec)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [68]:
!pip install -U langchain-text-splitters


In [18]:
# @title Task 19 — Build FAISS Vector Index
from langchain_community.vectorstores import FAISS

# TODO: Call FAISS.from_documents(chunks, embeddings)
print('Building FAISS index...')
vector_db = FAISS.from_documents(chunks, embeddings)  # replace

results = vector_db.similarity_search('How does attention mechanism work?', k=2)
print(f"Vectors indexed: {vector_db.index.ntotal}")
print(f"Top result snippet: {results[0].page_content[:150]}")

Building FAISS index...
Vectors indexed: 665
Top result snippet: are meaningful to humans. For example, some attention heads can attend mostly to the next word, while others mainly attend from verbs to their direct 


In [6]:
!pip install -U langchain-classic langchain-huggingface langchain-community

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

# 1. RE-DEFINE THE MODEL (This fixes the NameError)
MODEL_ID = 'facebook/opt-125m'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

print("Loading model into VRAM...")
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)

# 2. SETUP THE PIPELINE
tok_opt = AutoTokenizer.from_pretrained(MODEL_ID)
hf_pipe = pipeline(
    'text-generation',
    model=model_4bit,
    tokenizer=tok_opt,
    max_new_tokens=128,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipe)

# 3. ASSEMBLE THE QA CHAIN
# Note: Ensure the 'vector_db' cell was run recently too!
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True
)

# 4. RUN QUERY
result = qa_chain.invoke({'query': 'What is the Transformer architecture?'})
print(f"\nAnswer: {result['result']}")

Loading model into VRAM...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Architecture[edit]
All transformers have the same primary components:

Terminology[edit]
The transformer architecture, being modular, allows variations. Several common variations are described here.[60]

Transformer encoder with norm-first and norm-last
Transformer decoder with norm-first and norm-last
Block diagram for the full transformer architectureSchematic object hierarchy for the full transformer architecture, in object-oriented programming styleThe final points of detail are the residual connections and layer normalization, (denoted as "LayerNorm", or "LN" in the following), which while conceptually unnecessary, are necessary for numerical stability and convergence.

Question: What is the Transformer architecture?
Helpful Answer:
The Transformer architecture is a modular, modular, and scalable architecture. 

In [ ]:
# @title 🔍 Section 4 Grader { display-mode: "form" }
from langchain_community.vectorstores import FAISS as _FAISS
from langchain.chains import RetrievalQA as _RQA

_s4_passed = _s4_total = 0

def run_test(label, fn):
    global _s4_passed, _s4_total
    _s4_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s4_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 16 — document loading')
run_test('docs is a list with 2+ entries', lambda: (_ for _ in ()).throw(AssertionError()) if not (isinstance(docs,list) and len(docs)>=2) else None)
def _t16b():
    assert hasattr(docs[0],'page_content') and len(docs[0].page_content)>500
run_test('documents have substantial page_content', _t16b)

print('Task 17 — chunking')
def _t17a():
    assert isinstance(chunks,list) and len(chunks)>len(docs)
def _t17b():
    assert all(len(c.page_content)<=700 for c in chunks), 'some chunks too long'
run_test('chunks > docs (splitting happened)', _t17a)
run_test('all chunks within size limit', _t17b)

print('Task 18 — embeddings')
def _t18a():
    assert embeddings is not None
    v = embeddings.embed_query('test')
    assert len(v)==384, f'expected 384-dim, got {len(v)}'
def _t18b():
    import numpy as np
    v = embeddings.embed_query('hello')
    assert abs(np.linalg.norm(v)-1.0)<0.01, 'vectors not normalized'
run_test('embeddings produce 384-dim vectors', _t18a)
run_test('vectors are L2-normalized', _t18b)

print('Task 19 — FAISS index')
def _t19a():
    assert isinstance(vector_db,_FAISS) and vector_db.index.ntotal==len(chunks)
def _t19b():
    r = vector_db.similarity_search('attention', k=1)
    assert len(r)==1 and len(r[0].page_content)>10
run_test(f'index contains {len(chunks)} vectors', _t19a)
run_test('similarity_search returns relevant result', _t19b)

print('Task 20 — RetrievalQA chain')
def _t20a():
    assert isinstance(qa_chain,_RQA)
def _t20b():
    assert result is not None and 'result' in result and 'source_documents' in result
    assert len(result['result'])>5
run_test('qa_chain is a RetrievalQA instance', _t20a)
run_test('invoke returns result + source_documents', _t20b)

print(f"\nSection 4: {_s4_passed}/{_s4_total} tests passed")

---
## ✏️ Section 5: Agents with CrewAI
**30 min | Tasks 21–25** — Define agents, give them roles and tools, run a multi-agent crew.

CrewAI builds on top of LangChain. An `Agent` has a **role**, **goal**, and **backstory**.
A `Task` tells an agent what to produce. A `Crew` orchestrates multiple agents.


In [20]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# 1. RE-INITIALIZE EMBEDDINGS
# (Required to build the Vector DB)
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 2. RE-BUILD VECTOR DB
# This fixes the "NameError: name 'vector_db' is not defined"
# Note: This assumes 'chunks' is still in memory from Task 17.
print("Building FAISS index...")
vector_db = FAISS.from_documents(chunks, embeddings)

# 3. ASSEMBLE THE RETRIEVAL QA CHAIN
# Now that vector_db exists, we can create the retriever
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

print("Assembling QA Chain...")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True
)

# 4. RUN THE FINAL QUERY
print("Running Query...")
query = "What is the Transformer architecture?"
result = qa_chain.invoke({'query': query})

print("-" * 30)
print(f"Answer: {result['result']}")
print(f"Sources used: {len(result['source_documents'])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS index...


Both `max_new_tokens` (=128) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assembling QA Chain...
Running Query...
------------------------------
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Architecture[edit]
All transformers have the same primary components:

Terminology[edit]
The transformer architecture, being modular, allows variations. Several common variations are described here.[60]

Transformer encoder with norm-first and norm-last
Transformer decoder with norm-first and norm-last
Block diagram for the full transformer architectureSchematic object hierarchy for the full transformer architecture, in object-oriented programming styleThe final points of detail are the residual connections and layer normalization, (denoted as "LayerNorm", or "LN" in the following), which while conceptually unnecessary, are necessary for numerical stability and convergence.

Question: What is the Transformer architecture?
Helpful Answer:
The Trans

In [ ]:
# @title Task 22 — Define a Researcher Agent
# An Agent has: role, goal, backstory, llm, and optional tools.

# TODO: Create an Agent named researcher with:
#   role      = 'AI Research Analyst'
#   goal      = 'Summarise key facts about a given AI topic clearly and concisely'
#   backstory = (write 1-2 sentences describing the agent's expertise)
#   llm       = crew_llm
#   verbose   = True
#   allow_delegation = False
researcher = None  # replace with Agent(...)

print(f"Agent role: {researcher.role}")

In [ ]:
# @title Task 23 — Define a Writer Agent
# TODO: Create a second Agent named writer with:
#   role      = 'Technical Writer'
#   goal      = 'Turn research notes into a clear, engaging paragraph for a general audience'
#   backstory = (write your own)
#   llm       = crew_llm
#   verbose   = True
#   allow_delegation = False
writer = None  # replace with Agent(...)

print(f"Agent role: {writer.role}")

In [ ]:
# @title Task 24 — Define Tasks for Each Agent
# A Task has: description, expected_output, and agent.

TOPIC = 'Retrieval-Augmented Generation (RAG)'

# TODO: Create research_task with:
#   description     = f'Research the following topic and list 5 key facts: {TOPIC}'
#   expected_output = 'A bullet-point list of 5 factual statements'
#   agent           = researcher
research_task = None  # replace

# TODO: Create write_task with:
#   description     = 'Using the research notes, write a 3-sentence summary suitable for a blog post'
#   expected_output = 'A 3-sentence paragraph'
#   agent           = writer
write_task = None  # replace

print(f"Tasks defined: research={research_task is not None}, write={write_task is not None}")

In [ ]:
# @title Task 25 — Assemble and Run the Crew
# TODO: Create a Crew with:
#   agents  = [researcher, writer]
#   tasks   = [research_task, write_task]
#   process = Process.sequential      (researcher goes first, hands off to writer)
#   verbose = True
crew = None  # replace

print('Running crew... (may take 1-2 min)')
crew_result = crew.kickoff()
print("\n===== CREW OUTPUT =====")
print(crew_result)

In [22]:
# @title 🔍 Section 5 Grader { display-mode: "form" }
from crewai import Agent as _Agent, Task as _Task, Crew as _Crew

_s5_passed = _s5_total = 0

def run_test(label, fn):
    global _s5_passed, _s5_total
    _s5_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s5_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 21 — crew_llm')
def _t21a():
    assert crew_llm is not None
def _t21b():
    out = crew_llm('Say yes:')
    assert isinstance(out,str) and len(out)>0
run_test('crew_llm is not None', _t21a)
run_test('crew_llm generates string output', _t21b)

print('Task 22 — researcher agent')
def _t22a():
    assert isinstance(researcher,_Agent), f'got {type(researcher)}'
def _t22b():
    assert researcher.role and len(researcher.role)>3
    assert researcher.goal and len(researcher.goal)>10
    assert researcher.backstory and len(researcher.backstory)>10
run_test('researcher is a CrewAI Agent', _t22a)
run_test('role, goal, backstory all populated', _t22b)

print('Task 23 — writer agent')
def _t23a():
    assert isinstance(writer,_Agent)
def _t23b():
    assert writer.role != researcher.role, 'writer and researcher must have different roles'
run_test('writer is a CrewAI Agent', _t23a)
run_test('writer has a distinct role from researcher', _t23b)

print('Task 24 — tasks')
def _t24a():
    assert isinstance(research_task,_Task) and isinstance(write_task,_Task)
def _t24b():
    assert research_task.agent == researcher
    assert write_task.agent == writer
run_test('both tasks are CrewAI Task instances', _t24a)
run_test('tasks assigned to correct agents', _t24b)

print('Task 25 — crew execution')
def _t25a():
    assert isinstance(crew,_Crew) and len(crew.agents)==2
def _t25b():
    assert crew_result is not None and len(str(crew_result))>20
run_test('crew has 2 agents and 2 tasks', _t25a)
run_test('crew_result is non-empty', _t25b)

print(f"\nSection 5: {_s5_passed}/{_s5_total} tests passed")

Task 21 — crew_llm
  ❌ FAIL  crew_llm is not None
         -> name 'crew_llm' is not defined
  ❌ FAIL  crew_llm generates string output
         -> name 'crew_llm' is not defined
Task 22 — researcher agent
  ❌ FAIL  researcher is a CrewAI Agent
         -> name 'researcher' is not defined
  ❌ FAIL  role, goal, backstory all populated
         -> name 'researcher' is not defined
Task 23 — writer agent
  ❌ FAIL  writer is a CrewAI Agent
         -> name 'writer' is not defined
  ❌ FAIL  writer has a distinct role from researcher
         -> name 'writer' is not defined
Task 24 — tasks
  ❌ FAIL  both tasks are CrewAI Task instances
         -> name 'research_task' is not defined
  ❌ FAIL  tasks assigned to correct agents
         -> name 'research_task' is not defined
Task 25 — crew execution
  ❌ FAIL  crew has 2 agents and 2 tasks
         -> name 'crew' is not defined
  ❌ FAIL  crew_result is non-empty
         -> name 'crew_result' is not defined

Section 5: 0/10 tests passed


---
## 🏆 Final Grader
Run this cell after completing all sections to see your overall score.


In [21]:
# @title 🏆 Final Cumulative Grader { display-mode: "form" }
print('=' * 50)
print('       FINAL LAB SCORE REPORT')
print('=' * 50)

sections = [
    ('Section 1 — Foundations',       _s1_passed, _s1_total),
    ('Section 2 — Quantization',      _s2_passed, _s2_total),
    ('Section 3 — Inference',         _s3_passed, _s3_total),
    ('Section 4 — RAG with FAISS',    _s4_passed, _s4_total),
    ('Section 5 — Agents (CrewAI)',   _s5_passed, _s5_total),
]

grand_passed = grand_total = 0
for name, p, t in sections:
    bar   = '█' * p + '░' * (t - p)
    grand_passed += p
    grand_total  += t
    status = '✅' if p == t else ('⚠️ ' if p >= t//2 else '❌')
    print(f'  {status}  {name:<35} {p}/{t}  [{bar}]')

pct = int(grand_passed / grand_total * 100) if grand_total else 0
print('=' * 50)
print(f'  TOTAL: {grand_passed}/{grand_total} tests passed  ({pct}%)')
print('=' * 50)

if pct == 100:
    print('  🎉 Perfect score! All tasks complete.')
elif pct >= 80:
    print('  👍 Great work! Review any failing sections above.')
elif pct >= 50:
    print('  📖 Good progress. Revisit failing tasks before moving on.')
else:
    print('  🔧 Keep going — run section graders first, then rerun this cell.')

       FINAL LAB SCORE REPORT


NameError: name '_s1_passed' is not defined

---
## What You Built

| Section | Packages used | Key concept |
|---|---|---|
| 1 Foundations | `numpy`, `torch`, `transformers` | BPE, softmax, attention, MoE |
| 2 Quantization | `bitsandbytes`, `transformers` | NF4 4-bit, VRAM budgeting |
| 3 Inference | `ctransformers` | Temperature, top-p, top-k, prompting |
| 4 RAG | `langchain`, `faiss-gpu`, `sentence-transformers` | Embed → Index → Retrieve → Generate |
| 5 Agents | `crewai` | Multi-agent roles, tasks, sequential crew |
